# Method 작성과 사용 예제

간단한 그룹별 조회 Method를 작성하고 실행합니다. 이어서 코드를 수정하고, 여러 조회를 조합한 분석과 DoWhy 분석을 만들어봅니다.

연습에는 준비된 예제 데이터를 사용합니다. 마지막에는 같은 Method를 실제 Cube 데이터에 적용합니다.

## 1. 준비

In [46]:
from pathlib import Path
import sys

repository = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/decision_layer").is_dir()
)
if str(repository) not in sys.path:
    sys.path.insert(0, str(repository))

import pandas as pd
from IPython.display import display

from decision_layer.dev import MethodSession
from decision_layer.methods import (
    Dimension, Entity, Estimate, Filter, Measure, Refused, Scope, Table, method,
)
import examples.methods.patterns.teaching_data as example_data


def show_table(data):
    if isinstance(data, dict):
        data = [{"field": key, "value": value} for key, value in data.items()]
    display(pd.DataFrame(data))

## 2. 그룹별 조회 Method 만들기

예제로 지표를 그룹별로 조회하고 값이 큰 순서로 상위 `limit`개를 반환하는 Method를 만듭니다.

In [39]:
@method("workshop.by_group", version="1.0.0", interpretation="descriptive")
async def my_by_group(ctx, *, measure: Measure, group: Dimension,
                          limit: int = 3) -> Table:
    data = await ctx.aggregate(
        measures={"value": measure},
        dimensions={"group": group},
        order=[(measure, "desc")],
        limit=limit,
    )
    return Table.from_dataset(data, require_non_empty=True)

`ctx`는 함수가 받는 데이터 조회 객체입니다. 함수 안에서 `ctx.aggregate(...)`로 `measure` 지표를 `group`별로 조회합니다.

- `measures={"value": measure}`: `measure` 지표를 조회하고 결과 열을 `value`로 부릅니다.
- `dimensions={"group": group}`: `group` 기준으로 나누고 결과 열을 `group`으로 부릅니다.
- `order`, `limit`: 지표 값이 큰 순서로 상위 `limit`개를 가져옵니다.

### 요청을 Method 입력으로 바꾸기

예: “2001년 2월 store 364의 영수증 수를 department별 상위 3개 보여줘.”

AI에게 요청하면 semantic layer에서 필요한 지표와 분류 기준을 찾아 입력을 채웁니다. 이 노트북에서는 아래 실행 코드에 같은 값을 직접 넣어봅니다.

- `measure`: 영수증 수 지표 `cube://journey/transaction/receipts`
- `group`: department 분류 기준 `cube://journey/product/department`
- `limit`: 3
- `scope`: 2001년 2월, 거래 날짜 기준, store 364 조건

`cube://journey/transaction/receipts`는 `journey` 연결의 `transaction` 모델에 있는 `receipts` 지표를 뜻합니다. 연습 코드에서는 준비된 예제 데이터의 `cube://practice/observations/value`를 사용합니다.

### 등록하고 실행

함수를 등록한 뒤 같은 인자 이름으로 호출합니다. `scope`에는 기간과 날짜 dimension을 지정합니다. 연습 응답의 값은 `30, 20, 10`이고 조회는 한 번입니다.

In [29]:
group_session = await example_data.example_session("by_group")
group_session.register(my_by_group)

trial = await group_session.run(
    my_by_group,
    measure="cube://practice/observations/value",
    group="cube://practice/observations/group",
    scope=Scope(
        ("2026-02-01", "2026-02-28"),
        "cube://practice/observations/date",
    ),
    limit=3,
)

assert trial.result.status == "success"
assert [row["value"] for row in trial.result.primary.data] == [30, 20, 10]
assert len(trial.attempts) == 1
show_table(trial.result.primary.data)

group_session.close()

,value,group
0,30,A
1,20,B
2,10,C


위 실행 셀을 실행하면 `group_session.run(...)`이 `ctx`를 만들어 `my_by_group`에 넣습니다. 함수 안의 `ctx.aggregate(...)`가 실제 조회를 실행하고, 결과 표에 `30, 20, 10`을 반환합니다. `scope`의 기간과 조건도 이 조회에 적용됩니다.

### 조회 기록 확인

요청한 기간과 필터와 실행 상태를 확인합니다.

In [ ]:
show_table([
    {
        "query": index,
        "label": attempt.label,
        "status": attempt.status,
        "period": attempt.spec.time.date_range,
        "filters": [item.model_dump() for item in attempt.spec.filters],
    }
    for index, attempt in enumerate(trial.attempts, start=1)
])

## 3. 코드 수정: 순위 추가

조회 결과에 `rank`를 추가합니다. `data.records()` 다음 부분이 바뀝니다. 수정한 함수는 같은 Method 이름과 새 버전 `1.1.0`을 사용합니다.

In [ ]:
@method("workshop.by_group", version="1.1.0", interpretation="descriptive")
async def my_ranked_by_group(ctx, *, measure: Measure, group: Dimension,
                          limit: int = 3) -> Table:
    data = await ctx.aggregate(
        measures={"value": measure},
        dimensions={"group": group},
        order=[(measure, "desc")],
        limit=limit,
    )
    rows = data.records()
    for rank, row in enumerate(rows, start=1):
        row["rank"] = rank
    return Table(rows)

### 수정한 함수 실행

`replace=True`로 등록된 구현을 교체합니다. 결과에 순위 `1, 2, 3`이 추가됩니다.

In [ ]:
ranked_session = await example_data.example_session("by_group")
ranked_session.register(my_by_group)
ranked_session.register(my_ranked_by_group, replace=True)

edited_trial = await ranked_session.run(
    my_ranked_by_group,
    measure="cube://practice/observations/value",
    group="cube://practice/observations/group",
    scope=Scope(
        ("2026-02-01", "2026-02-28"),
        "cube://practice/observations/date",
    ),
    limit=3,
)

assert edited_trial.result.status == "success"
assert [row["rank"] for row in edited_trial.result.primary.data] == [1, 2, 3]
assert [row["value"] for row in edited_trial.result.primary.data] == [30, 20, 10]
show_table(edited_trial.result.primary.data)

ranked_session.close()

## 4. 여러 조회 조합: Peer Comparison

한 함수에서 세 번 조회하고, 받은 값을 하나의 표로 합칩니다.

| 조회 | 대상 |
| --- | --- |
| subject | 그룹 X의 대상 A |
| peers | 그룹 X에서 A를 제외한 집단 |
| overall | 전체에서 A를 제외한 집단 |

`ctx.scope.filters`는 모든 조회에 적용되는 공통 조건입니다. 아래 함수는 비교 집단까지 대상 A로 제한되지 않도록 이 조건을 확인합니다.

In [ ]:
@method(
    "workshop.peer",
    version="1.0.0",
    interpretation="descriptive",
)
async def my_peer(
    ctx,
    *,
    measure: Measure,
    subject: Dimension,
    category: Dimension,
    subject_value: str,
    category_value: str,
) -> Table:
    if any(item.member == subject for item in ctx.scope.filters):
        raise Refused("Keep the subject out of shared filters")

    same_category = Filter(
        member=category,
        operator="equals",
        values=[category_value],
    )
    selected = Filter(
        member=subject, 
        operator="equals", 
        values=[subject_value]
    )
    excluded = Filter(member=subject, operator="notEquals", values=[subject_value])

    subject_data = await ctx.aggregate(
        measures={"value": measure},
        filters=[same_category, selected],
        label="subject",
    )
    peer_data = await ctx.aggregate(
        measures={"value": measure},
        filters=[same_category, excluded],
        label="peers",
    )
    overall_data = await ctx.aggregate(
        measures={"value": measure},
        filters=[excluded],
        label="overall",
    )

    subject_result = subject_data.scalar("value")
    peer_result = peer_data.scalar("value")
    overall_result = overall_data.scalar("value")

    return Table(
        [
            {
                "comparison": "peers",
                "subject": subject_result,
                "benchmark": peer_result,
                "difference": subject_result - peer_result,
            },
            {
                "comparison": "overall",
                "subject": subject_result,
                "benchmark": overall_result,
                "difference": subject_result - overall_result,
            },
        ],
        warnings=["Descriptive comparison; no causal claim or significance test."],
    )

### 등록하고 실행

대상 A와 그룹 X를 지정합니다. 준비된 응답은 `12, 8, 6`이므로 차이는 `4, 6`입니다.

In [ ]:
peer_session = await example_data.example_session("peer")
peer_session.register(my_peer)

peer_trial = await peer_session.run(
    my_peer,
    measure="cube://practice/observations/rate",
    subject="cube://practice/observations/subject",
    category="cube://practice/observations/group",
    scope=Scope(
        ("2026-02-01", "2026-02-28"),
        "cube://practice/observations/date",
    ),
    subject_value="A",
    category_value="X",
)

assert peer_trial.result.status == "success"
assert [row["difference"] for row in peer_trial.result.primary.data] == [4, 6]
assert len(peer_trial.attempts) == 3
show_table(peer_trial.result.primary.data)

peer_session.close()

### 세 조회 확인

각 조회의 label과 필터를 비교합니다. 직접 수정해보려면 결과에 `ratio`를 추가해보세요.

In [ ]:
show_table([
    {
        "query": index,
        "label": attempt.label,
        "status": attempt.status,
        "filters": [item.model_dump() for item in attempt.spec.filters],
    }
    for index, attempt in enumerate(peer_trial.attempts, start=1)
])

## 5. DoWhy 회귀 조정

DoWhy의 `backdoor.linear_regression`으로 이진 처치의 `0→1` 차이를 조정합니다.

| 입력 | 의미 |
| --- | --- |
| outcome | 결과 지표 |
| treatment | 0 또는 1인 처치 |
| confounders | 처치 전 조정 변수 |
| unit | 분석 단위 |

데이터 조건을 검사해도 식별 가정이 검증되는 것은 아닙니다. 이 예제는 연관성 결과를 보고하며 신뢰구간을 제공하지 않습니다.

### 계산 함수 작성

데이터 검사 → 전처리 → 지원 범위 확인 → DoWhy 추정 순서입니다. 같은 코드를 [binary_regression.py](binary_regression.py)에서도 볼 수 있습니다.

In [ ]:
def estimate_binary_regression(frame, confounder_columns):
    """Estimate the 0-to-1 contrast under an explicitly assumed backdoor graph."""
    import networkx as nx
    import numpy as np
    import pandas as pd
    from dowhy import CausalModel
    from sklearn.linear_model import LogisticRegression

    # 1. Check the observation units, missing values and treatment groups.
    if (
        frame.empty
        or frame["unit"].isna().any()
        or frame["unit"].duplicated().any()
    ):
        raise Refused("Unique non-null analysis units are required")

    required_columns = ["outcome", "treatment", *confounder_columns]
    if not confounder_columns or frame[required_columns].isna().any().any():
        raise Refused(
            "Observed adjustment variables and a missing-data policy are required"
        )

    if (
        set(frame["treatment"].unique()) != {0, 1}
        or frame.groupby("treatment").size().min() < 30
    ):
        raise Refused(
            "This example requires binary treatment and at least 30 units per arm"
        )

    # 2. Bound encoding before allocating the matrix.
    encoded_width = sum(
        1 if pd.api.types.is_numeric_dtype(frame[column]) else max(frame[column].nunique() - 1, 0)
        for column in confounder_columns
    )
    if encoded_width == 0 or encoded_width > 100 or encoded_width * len(frame) > 2_000_000:
        raise Refused("Adjustment encoding is empty or exceeds the matrix bound")
    encoded = pd.get_dummies(
        frame[confounder_columns],
        drop_first=True,
        dtype=float,
    )
    if (
        not len(encoded.columns)
        or len(encoded.columns) > 100
        or encoded.size > 2_000_000
    ):
        raise Refused("Adjustment encoding is empty or exceeds the matrix bound")

    encoded.columns = [f"adjustment_{i}" for i in range(len(encoded.columns))]

    if (
        not np.isfinite(frame["outcome"].to_numpy(dtype=float)).all()
        or not np.isfinite(encoded.to_numpy()).all()
    ):
        raise Refused("Finite outcomes and adjustment variables are required")

    observations = pd.concat(
        [
            frame[["outcome", "treatment"]].reset_index(drop=True),
            encoded.reset_index(drop=True),
        ],
        axis=1,
    )

    predictor_columns = ["treatment", *encoded.columns]
    predictors = observations[predictor_columns].to_numpy(dtype=float)
    matrix = np.column_stack([np.ones(len(frame)), predictors])

    if (
        len(frame) <= matrix.shape[1]
        or np.linalg.matrix_rank(matrix) < matrix.shape[1]
    ):
        raise Refused(
            "The regression needs full-rank predictors and residual degrees of freedom"
        )

    # 3. Diagnose observed treatment support. This does not prove positivity.
    propensity = LogisticRegression(
        max_iter=1000,
        random_state=7,
    ).fit(encoded, frame["treatment"])
    probability = propensity.predict_proba(encoded)[:, 1]

    if np.any(probability < 0.05) or np.any(probability > 0.95):
        raise Refused(
            "Observed support fails this example's diagnostic; not a positivity proof"
        )

    # 4. Declare the assumed DAG and identify its backdoor estimand.
    edges = [("treatment", "outcome")]
    edges += [
        (column, target)
        for column in encoded.columns
        for target in ("treatment", "outcome")
    ]
    model = CausalModel(
        data=observations,
        treatment="treatment",
        outcome="outcome",
        graph=nx.DiGraph(edges),
    )
    estimand = model.identify_effect(proceed_when_unidentifiable=False)

    if not (estimand.estimands.get("backdoor") or {}).get("estimand"):
        raise Refused("No backdoor estimand is identified under the assumed graph")

    # 5. Fit the explicit binary regression contrast and a placebo diagnostic.
    estimate = model.estimate_effect(
        estimand,
        method_name="backdoor.linear_regression",
        control_value=0,
        treatment_value=1,
        target_units="ate",
        confidence_intervals=False,
    )
    placebo = model.refute_estimate(
        estimand,
        estimate,
        method_name="placebo_treatment_refuter",
        placebo_type="permute",
        num_simulations=20,
        random_state=7,
    )

    # 6. Return the estimate with its analysis settings and interpretation limits.
    return {
        "value": float(estimate.value),
        "estimator": "backdoor.linear_regression",
        "contrast": [0, 1],
        "target": "ate",
        "n_units": len(frame),
        "graph_edges": edges,
        "estimand": str(estimand),
        "placebo_difference": float(placebo.new_effect),
        "seed": 7,
        "support": {
            "min": float(probability.min()),
            "max": float(probability.max()),
        },
        "causal_claim": "not_established",
        "uncertainty": "not_reported",
    }

### Method로 연결

`ctx.entities(...)`로 `unit`별 데이터를 가져온 뒤, `ctx.compute(...)`로 위 계산 함수를 실행합니다.

In [ ]:
@method(
    "workshop.binary_regression",
    version="1.0.0",
    interpretation="associational",
    execution="dataframe",
    requires_packages={
        "dowhy": "==0.13",
        "numpy": ">=1.26,<2",
        "pandas": ">=2.2,<3",
        "scikit-learn": ">=1.2,<1.7",
    },
)
async def my_binary_regression(
    ctx,
    *,
    outcome: Measure,
    treatment: Dimension,
    confounders: list[Dimension | Measure],
    unit: Entity,
) -> Estimate:
    # Extract governed entity data, then run ordinary Python calculation code.
    observations = await ctx.entities(
        entity=unit,
        measures={"outcome": outcome},
        dimensions={
            "treatment": treatment,
            "confounders": confounders,
        },
    )
    payload = await ctx.compute(
        estimate_binary_regression,
        observations.frame,
        observations.columns("confounders"),
    )

    return Estimate(
        payload["value"],
        details={
            **payload,
            "assumptions_status": "not_verified",
        },
        warnings=[
            "Unverified assumptions and placebo diagnostics do not establish "
            "causality; no interval is reported."
        ],
    )

### 등록하고 실행

결과, 처치, 조정 변수, 분석 단위를 연결합니다. 예제 데이터의 생성식은 `Y = 2*T + 3*W + noise`이며 기대값은 약 `2`입니다.

In [ ]:
regression_session = await example_data.example_session("dowhy")
regression_session.register(my_binary_regression)

dowhy_trial = await regression_session.run(
    my_binary_regression,
    outcome="cube://practice/observations/value",
    treatment="cube://practice/observations/treated",
    confounders=["cube://practice/observations/baseline"],
    unit="cube://practice/observations/id",
    scope=Scope(
        ("2026-02-01", "2026-02-28"),
        "cube://practice/observations/date",
    ),
)

assert dowhy_trial.result.status == "success"
assert abs(dowhy_trial.result.primary.data["value"] - 2) < 0.15
show_table(dowhy_trial.result.primary.data)

regression_session.close()


### 일반 Python에서도 사용

같은 계산 함수에 DataFrame을 직접 전달합니다. Method 실행과 같은 값이 나오는지 확인합니다.

In [ ]:
frame = example_data.binary_regression_frame()
standalone_result = estimate_binary_regression(frame, ["baseline"])

assert abs(standalone_result["value"] - 2) < 0.15
assert standalone_result["value"] == dowhy_trial.result.primary.data["value"]
show_table(standalone_result)


### 실패 확인

분석 단위가 중복된 연습 데이터를 넣습니다. Method는 추정을 거절하고 이유를 반환합니다.

In [ ]:
refusal_session = await example_data.duplicate_unit_session()
refusal_session.register(my_binary_regression)

refused_trial = await refusal_session.run(
    my_binary_regression,
    outcome="cube://practice/observations/value",
    treatment="cube://practice/observations/treated",
    confounders=["cube://practice/observations/baseline"],
    unit="cube://practice/observations/id",
    scope=Scope(
        ("2026-02-01", "2026-02-28"),
        "cube://practice/observations/date",
    ),
)

assert refused_trial.result.status == "refused"
show_table([
    {"status": refused_trial.result.status, "reason": warning}
    for warning in refused_trial.result.warnings
])
refusal_session.close()


## 6. 실제 Cube에서 사용

연결 셀의 `RUN_LIVE`를 `True`로 바꾸세요. 기본 URL과 secret은 로컬 샘플 설정입니다. 실제 catalog에서 모델과 항목을 가리키는 경로를 확인한 뒤 위 함수를 그대로 실행합니다.

In [ ]:
import os
from decision_layer.semantic.credentials import ServiceCredentials

RUN_LIVE = False
live = None

if RUN_LIVE:
    live = await MethodSession.cube(
        os.environ.get("DL_JOURNEY_CUBE_URL", "http://127.0.0.1:4000/cubejs-api/v1"),
        ServiceCredentials(
            os.environ.get(
                "DL_JOURNEY_CUBE_SECRET",
                "local-example-secret-change-me-0123456789",
            ),
            (),
        ),
        instance="journey",
    )

show_table([{
    "status": "connected" if live is not None else "skipped",
    "reason": "Cube connected" if live is not None else "Set RUN_LIVE = True in this connection cell",
}])

### 객체 확인

연결한 Cube에서 receipts 지표와 department를 찾습니다. catalog는 사용 가능한 객체 목록이며, Method 안에서도 `ctx.catalog`로 확인할 수 있습니다.

In [ ]:
if live is not None:
    objects = [
        *live.catalog.search(kind="measure", text="receipts"),
        *live.catalog.search(kind="dimension", text="department"),
    ]
    show_table([
        {"title": obj.title, "kind": obj.kind, "path": obj.ref}
        for obj in objects
    ])
else:
    show_table([{"status": "skipped", "reason": "Enable the Cube connection above"}])

### 그룹별 조회

영수증 수를 department별로 조회합니다. `limit`이나 기간을 바꿔보세요.

In [ ]:
if live is not None:
    live.register(my_by_group, replace=True)
    live_group_trial = await live.run(
        my_by_group,
        measure="cube://journey/transaction/receipts",
        group="cube://journey/product/department",
        scope=Scope(
            ("2001-01-01", "2001-06-30"),
            "cube://journey/transaction/analysis_date",
        ),
        limit=3,
    )
    show_table(live_group_trial.result.primary.data)
else:
    show_table([{"status": "skipped", "reason": "Enable the Cube connection above"}])

### 집단 비교

GROCERY의 store 364를 동종 집단과 전체 집단과 비교합니다. 실제 존재하는 다른 store나 department로 바꿔보세요.

In [ ]:
if live is not None:
    live.register(my_peer, replace=True)
    live_peer_trial = await live.run(
        my_peer,
        measure="cube://journey/transaction/coupon_line_rate",
        subject="cube://journey/transaction/store",
        category="cube://journey/product/department",
        scope=Scope(
            ("2001-01-01", "2001-06-30"),
            "cube://journey/transaction/analysis_date",
        ),
        subject_value="364",
        category_value="GROCERY",
    )
    show_table(live_peer_trial.result.primary.data)
    show_table([
        {"query": index, "label": attempt.label, "status": attempt.status}
        for index, attempt in enumerate(live_peer_trial.attempts, start=1)
    ])
else:
    show_table([{"status": "skipped", "reason": "Enable the Cube connection above"}])

### 연결 정리

로컬 session을 닫습니다. 서버에 제공할 Method는 검토한 Python 파일에 저장하고 서버에서 등록합니다.

In [ ]:
if live is not None:
    live.close()
    live = None

show_table([{"status": "closed"}])